In [1]:
import pandas as pd
# patient 210 is an absolute hero
df = pd.read_csv("heart_failure_clinical_records_dataset.csv")


In [2]:
round(
    (
        (df["ejection_fraction"] >= 26.25) &
        (df["ejection_fraction"] <= 49.91)
    ).mean() * 100,
    2
) 
# From this part of code we concluded that 60.2 percent of patients fall into standard deviation range.

np.float64(60.2)

In [3]:
med = df["ejection_fraction"].median()
# df[["ejection_fraction","DEATH_EVENT"]].sort_values("ejection_fraction")
df["EF_group"] = pd.cut(
    df["ejection_fraction"],
    bins=[0,38,float("inf")],
    labels=["Low EF","High EF"],
    right=False
)
df.groupby("EF_group")["DEATH_EVENT"].mean()
# Basically Lower Ejection Fraction was associated with higher mortality rate in this dataset.
# Let's dive into that more:


EF_group
Low EF     0.408451
High EF    0.242038
Name: DEATH_EVENT, dtype: float64

In [4]:
df["EF_percentage"] = pd.cut(
    df["ejection_fraction"],
    bins=[10, 20, 30, 40, 50, 60, 70, 80, 90],
    right=False,
    labels=["10%", "20%", "30%", "40%", "50%", "60%", "70%", "80%"]
)

result = (
    df.groupby("EF_percentage")["DEATH_EVENT"]
    .agg(
        patients="count",
        death_count="sum",
        death_rate="mean",
    )
    .assign(
        survival_rate=lambda x: 1 - x["death_rate"],
    )
    .assign(
        survival_rate_changes=lambda x:
            x["survival_rate"].pct_change() * 100
    )
)

result["survival_rate"] = result["survival_rate"].map(
    lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "N/A"
)

result["survival_rate_changes"] = result["survival_rate_changes"].map(
    lambda x: f"{x:+.2f}%" if pd.notna(x) else "N/A"
)

result

,patients,death_count,death_rate,survival_rate,survival_rate_changes
EF_percentage,,,,,
10%,5,4,0.800000,20.00%,N/A
20%,54,34,0.629630,37.04%,+85.19%
30%,123,35,0.284553,71.54%,+93.17%
40%,57,9,0.157895,84.21%,+17.70%
50%,24,7,0.291667,70.83%,-15.89%
60%,34,6,0.176471,82.35%,+16.26%
70%,1,1,1.000000,0.00%,-100.00%
80%,1,0,0.000000,100.00%,+inf%


In [5]:
#  this results shows us that EF alone does not guarantee a clear prediction of survival chance 
# as you can see from 10% EF to 50% EF we have a continious increase in survival rate
# but from 50% to 60% we have a decrease in survival rate. it could be because of insufficent 
# data from dataset or something else but due to my lack of medical knowledge i can not make a
# statement on this matter.